# 04x -- Manual Dynasty Rankings (DynastySharks + FantasyPros)

**Purpose:** Ingest hand-extracted dynasty sheets into the section-04 two-layer
model. Each sheet = one (source, format); source-specific metrics land as
`metric_key` rows so the schema never changes.

**Input:** `data/raw/DynastyRankings_2026_ManualExtraction.xlsx`. Identity via the
shared `etl.resolve_dynasty_crosswalk`; load via `etl.load_replace_partition`.
Run with CWD = repo root.

In [ ]:
# ---- Setup & Config ---------------------------------------------------------
import re
from dataclasses import dataclass, field
from datetime import date
from pathlib import Path

import pandas as pd

import sys
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG


@dataclass
class ManualDynastyConfig:
    """Manual dynasty-ranking ingest config."""
    xlsx: str = "data/raw/DynastyRankings_2026_ManualExtraction.xlsx"
    sheets: dict = field(default_factory=lambda: {       # sheet -> (source_name, format)
        "SF PPR-DynastySharks":        ("DynastySharks", "SF"),
        "SF TE Premium-DynastySharks": ("DynastySharks", "TEPP"),
        "SF PPR-FantasyPros":          ("FantasyPros",   "SF"),
        "IDP-FantasyPros":             ("FantasyPros",   "IDP"),
    })
    metric_maps: dict = field(default_factory=lambda: {  # per-source numeric metric_key -> header
        "DynastySharks": {"adp": "ADP", "proj_1yr": "1yr. Proj", "proj_3yr": "3yr. Proj",
                          "proj_5yr": "5yr. Proj", "proj_10yr": "10yr. Proj", "ds_value": "3D Value +"},
        "FantasyPros": {"best": "BEST", "worst": "WORST", "avg": "AVG.", "stddev": "STD.DEV"},
    })
    text_maps: dict = field(default_factory=lambda: {
        "DynastySharks": {"analysis": "DS Analysis"}, "FantasyPros": {},
    })
    crosswalk: str = "dim_dynasty_crosswalk"
    fact_rank: str = "fact_dynasty_rankings"
    fact_metrics: str = "fact_dynasty_ranking_metrics"


MCFG = ManualDynastyConfig()
SNAPSHOT_DATE = date.today().isoformat()


def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", str(name).lower()).strip("-")

def _num(v):
    """Coerce a sheet cell to float, or None ('-', '', NaN, text)."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    s = str(v).replace(",", "").replace("%", "").strip()
    if s in ("", "-"):
        return None
    try:
        return float(s)
    except ValueError:
        return None

def _split_pos(tok):
    """'QB1' -> ('QB', 1); 'DE2' -> ('DE', 2); 'QB' -> ('QB', None)."""
    m = re.match(r"\s*([A-Za-z/]+)\s*(\d*)", str(tok))
    if not m:
        return str(tok), None
    return m.group(1), (int(m.group(2)) if m.group(2) else None)

print(f"snapshot_date={SNAPSHOT_DATE} | sheets={len(MCFG.sheets)}")

In [ ]:
# ---- Parse sheets -> backbone rows + metric rows ----------------------------
backbone, metrics = [], []
for sheet, (source_name, fmt) in MCFG.sheets.items():
    df = pd.read_excel(MCFG.xlsx, sheet_name=sheet)
    age_col = "Age" if "Age" in df.columns else ("AGE" if "AGE" in df.columns else None)
    mmap, tmap = MCFG.metric_maps.get(source_name, {}), MCFG.text_maps.get(source_name, {})

    for row in df.to_dict("records"):   # to_dict preserves non-identifier headers
        name = row.get("Name")
        if not isinstance(name, str) or not name.strip():
            continue
        pos_raw, pos_rank = _split_pos(row.get("Pos"))
        spid = _slug(name)
        uid = f"{source_name}|{spid}"
        rk = _num(row.get("RK"))
        age = _num(row.get(age_col)) if age_col else None
        backbone.append({
            "snapshot_date": SNAPSHOT_DATE, "source_name": source_name,
            "source_player_id": spid, "format": fmt, "source_uid": uid,
            "source_site": source_name, "player_name": name, "position_raw": pos_raw,
            "nfl_team": row.get("Team"), "age": age,
            "overall_rank": int(rk) if rk is not None else None, "positional_rank": pos_rank,
        })
        for mk, col in mmap.items():
            num = _num(row.get(col))
            if num is not None:
                metrics.append({"snapshot_date": SNAPSHOT_DATE, "source_name": source_name,
                                "source_player_id": spid, "format": fmt, "source_uid": uid,
                                "metric_key": mk, "metric_num": num, "metric_text": None})
        for mk, col in tmap.items():
            v = row.get(col)
            if isinstance(v, str) and v.strip():
                metrics.append({"snapshot_date": SNAPSHOT_DATE, "source_name": source_name,
                                "source_player_id": spid, "format": fmt, "source_uid": uid,
                                "metric_key": mk, "metric_num": None, "metric_text": v.strip()})

backbone = pd.DataFrame(backbone)
metrics = pd.DataFrame(metrics)
print("backbone by (source, format):")
print(backbone.groupby(["source_name", "format"]).size().to_string())
print(f"metric rows: {len(metrics)} | keys: {sorted(metrics['metric_key'].unique())}")

In [ ]:
# ---- Resolve identity per source (shared matcher) -> upsert -> rebuild review -
# Manual gsis overrides for nickname/full-name mismatches (fuzzy < 90):
MANUAL_OVERRIDES = {
    "DynastySharks": {
        "cameron-ward": "00-0040676", "cameron-skattebo": "00-0040715",
        "chigoziem-okonkwo": "00-0037809",
    },
    "FantasyPros": {
        "hollywood-brown": "00-0035662", "matt-hibner": "HIB564972",
        "foyesade-oluokun": "00-0034413", "cam-bynum": "00-0036629",
    },
}   # Daylan Smothers / Mark Fletcher: 2026 rookies absent from both registries -> unmatched

res_parts = []
for source_name in backbone["source_name"].unique():
    ident = (backbone[backbone["source_name"] == source_name]
             [["source_player_id", "player_name", "position_raw", "nfl_team"]]
             .drop_duplicates("source_player_id").copy())
    ident["source"] = source_name
    res = etl.resolve_dynasty_crosswalk(ident, data_dir=CFG.data_dir,
                                        overrides=MANUAL_OVERRIDES.get(source_name))
    print(f"{source_name}: {len(res)} ids | {res['match_method'].value_counts().to_dict()}")
    res_parts.append(res)

xpath = f"{CFG.data_dir}/{MCFG.crosswalk}.parquet"
xwalk = etl.upsert_dynasty_crosswalk(pd.concat(res_parts, ignore_index=True), xpath)
n_rev = etl.write_dynasty_review(xwalk, Path(CFG.data_dir) / "review" / "review_dynasty_crosswalk.csv")
print(f"[ok] crosswalk {len(xwalk)} rows / {xwalk['source'].nunique()} sources; {n_rev} unresolved")

In [ ]:
# ---- Attach FKs (via source_uid) + load both facts -------------------------
fk = xwalk.set_index("source_uid")
backbone["gsis_id"]    = backbone["source_uid"].map(fk["gsis_id"])
backbone["player_key"] = backbone["source_uid"].map(fk["player_key"])
backbone = backbone[["snapshot_date", "source_name", "source_player_id", "format", "source_uid",
                     "source_site", "player_name", "position_raw", "nfl_team", "age",
                     "overall_rank", "positional_rank", "gsis_id", "player_key"]]

bn = etl.load_replace_partition(backbone, f"{CFG.data_dir}/{MCFG.fact_rank}.parquet")
mn = etl.load_replace_partition(metrics,  f"{CFG.data_dir}/{MCFG.fact_metrics}.parquet")
print(f"[ok] +{len(backbone)} backbone rows (table now {bn}); +{len(metrics)} metric rows (table now {mn})")

In [ ]:
# ---- Validation -------------------------------------------------------------
bb = pd.read_parquet(f"{CFG.data_dir}/{MCFG.fact_rank}.parquet")
mt = pd.read_parquet(f"{CFG.data_dir}/{MCFG.fact_metrics}.parquet")
print("fact_dynasty_rankings by source x format:")
print(bb.groupby(["source_name", "format"]).size().to_string())
print("\nkey dupes:", bb.duplicated(["snapshot_date","source_name","source_player_id","format"]).sum())
man = bb[bb["source_name"].isin(["DynastySharks", "FantasyPros"])]
print(f"manual-source gsis {man['gsis_id'].notna().mean()*100:.1f}% | player_key {man['player_key'].notna().mean()*100:.1f}%")
print("\nmetric keys by source:")
print(mt.groupby(["source_name", "metric_key"]).size().to_string())